In [1]:
!pip install pandas scikit-learn joblib

In [3]:
import pandas as pd

# Load dataset
df = pd.read_csv("/content/drive/MyDrive/WA_Fn-UseC_-Telco-Customer-Churn.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
# Drop customerID (not useful)
df = df.drop("customerID", axis=1)

# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Fill missing values
df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)

# Convert target to binary
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

/tmp/ipykernel_7456/4138126776.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)


In [5]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Separate column types
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

# Pipelines
numeric_pipeline = Pipeline([
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine pipelines
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [7]:
from sklearn.linear_model import LogisticRegression

log_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

In [8]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier())
])

In [9]:
from sklearn.model_selection import GridSearchCV

log_param_grid = {
    "model__C": [0.1, 1, 10],
    "model__solver": ["liblinear"]
}

log_grid = GridSearchCV(log_pipeline, log_param_grid, cv=3, scoring="accuracy")
log_grid.fit(X_train, y_train)

print("Best Logistic Params:", log_grid.best_params_)

Best Logistic Params: {'model__C': 10, 'model__solver': 'liblinear'}


In [10]:
rf_param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10, 20]
}

rf_grid = GridSearchCV(rf_pipeline, rf_param_grid, cv=3, scoring="accuracy")
rf_grid.fit(X_train, y_train)

print("Best RF Params:", rf_grid.best_params_)

Best RF Params: {'model__max_depth': 10, 'model__n_estimators': 200}


In [11]:
from sklearn.metrics import accuracy_score, classification_report

# Logistic
y_pred_log = log_grid.predict(X_test)
print("Logistic Accuracy:", accuracy_score(y_test, y_pred_log))

# Random Forest
y_pred_rf = rf_grid.predict(X_test)
print("RF Accuracy:", accuracy_score(y_test, y_pred_rf))

print("\nClassification Report (RF):\n")
print(classification_report(y_test, y_pred_rf))

Logistic Accuracy: 0.8197303051809794
RF Accuracy: 0.8090844570617459

Classification Report (RF):

              precision    recall  f1-score   support

           0       0.84      0.91      0.88      1036
           1       0.68      0.52      0.59       373

    accuracy                           0.81      1409
   macro avg       0.76      0.72      0.73      1409
weighted avg       0.80      0.81      0.80      1409



In [12]:
import joblib

# Save best model (choose RF usually)
joblib.dump(rf_grid.best_estimator_, "churn_pipeline.pkl")

print(" Pipeline saved successfully!")

 Pipeline saved successfully!


In [13]:
# Load pipeline
loaded_model = joblib.load("churn_pipeline.pkl")

# Example prediction
sample = X_test.iloc[:1]
prediction = loaded_model.predict(sample)

print("Prediction:", prediction)

Prediction: [1]
